# zyBooks Study Notebook

This notebook fetches formatted zyBooks content from the **zyBooks Formatter** app and lets you study interactively with Gemini.

**Setup:** Run the first cell to configure your connection, then use the helper functions to fetch any section.

---

In [ ]:
import requests
import json
from IPython.display import Markdown, display, HTML
import os

# === CONFIGURATION ===
FORMATTER_URL = "https://zy-books-formatter.replit.app"
ZYBOOK_CODE = "CPPCS2520NguyenSpring2026"

# Option 1: Store your refresh token on the server (recommended, auto-refreshes)
#   Run this once: requests.post(f'{FORMATTER_URL}/api/token', json={'refresh_token': 'YOUR_TOKEN'})
# Option 2: Set auth token manually (expires in ~24h)
AUTH_TOKEN = ""   # Leave empty to use server-stored token

def fetch_section(chapter, section, display_markdown=True):
    """Fetch a formatted zyBooks section as markdown."""
    params = {
        "zybook_code": ZYBOOK_CODE,
        "chapter": chapter,
        "section": section
    }
    headers = {"Authorization": f"Bearer {AUTH_TOKEN}"} if AUTH_TOKEN else {}
    resp = requests.get(f"{FORMATTER_URL}/api/zybooks-markdown", params=params, headers=headers)
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {resp.json().get('error', 'Unknown error')}")
        return None
    data = resp.json()
    md = data['markdown']
    if display_markdown:
        display(Markdown(md))
    return md

def fetch_chapter(chapter, sections=None):
    """Fetch all sections of a chapter. Returns dict of section->markdown."""
    results = {}
    sec = 1
    if sections:
        for s in sections:
            md = fetch_section(chapter, s, display_markdown=False)
            if md:
                results[s] = md
    else:
        while True:
            md = fetch_section(chapter, sec, display_markdown=False)
            if md is None:
                break
            results[sec] = md
            sec += 1
    print(f"Fetched {len(results)} sections from Chapter {chapter}")
    return results

def save_chapter_md(chapter, sections_dict, filename=None):
    """Save chapter content to a markdown file."""
    fname = filename or f"Chapter_{chapter}.md"
    combined = "\n\n---\n\n".join(sections_dict.values())
    with open(fname, "w") as f:
        f.write(combined)
    print(f"Saved {fname} ({len(combined):,} characters)")
    return fname

def publish_to_notion(title, markdown, parent_page_id=None):
    """Publish markdown content as a new Notion page."""
    resp = requests.post(f"{FORMATTER_URL}/api/notion/send", json={
        "markdown": markdown,
        "title": title,
        "parentPageId": parent_page_id
    })
    if resp.status_code != 200:
        print(f"Error: {resp.json().get('error', 'Unknown error')}")
        return None
    result = resp.json()
    print(f"Published to Notion: {result.get('url', '(no URL returned)')}")
    return result

print("zyBooks Formatter connected!")
print(f"App: {FORMATTER_URL}")
print(f"Book: {ZYBOOK_CODE}")
status = requests.get(f"{FORMATTER_URL}/api/token/status").json()
if status.get("configured"):
    print(f"Server token: ✓ configured (expires in {status['expires_in_hours']}h)")
elif AUTH_TOKEN:
    print("Using manual AUTH_TOKEN")
else:
    print("⚠ No token configured! Set AUTH_TOKEN above or run: requests.post(f'{FORMATTER_URL}/api/token', json={'refresh_token': 'YOUR_TOKEN'})")

In [ ]:
import requests
FORMATTER_URL = "https://zy-books-formatter.replit.app"
# Seed the refresh token (one-time, server auto-refreshes after this)
# Replace YOUR_REFRESH_TOKEN and YOUR_ADMIN_KEY with your actual values
requests.post(f"{FORMATTER_URL}/api/token",
    json={"refresh_token": "YOUR_REFRESH_TOKEN"},
    headers={"X-Admin-Key": "YOUR_ADMIN_KEY"}
).json()

## Quick Start

Run any of these to get started:

In [ ]:
# Fetch a single section and display it
fetch_section(7, 5)

In [ ]:
# Fetch an entire chapter
ch7 = fetch_chapter(7)

# Display a specific section
if 1 in ch7:
    display(Markdown(ch7[1]))

In [ ]:
# Save chapter to a .md file (accessible in Colab's file browser)
ch7 = fetch_chapter(7)
save_chapter_md(7, ch7, "Chapter_7.md")

## Publish to Notion

Send your study notes or formatted content to Notion:

In [ ]:
# Publish a single section to Notion
md = fetch_section(7, 5, display_markdown=False)
publish_to_notion("7.5 LAB: Checker for integer string", md)

In [ ]:
# Publish an entire chapter to Notion
ch7 = fetch_chapter(7)
combined = "\n\n---\n\n".join(ch7.values())
publish_to_notion("Chapter 7: String Slicing", combined)

## Study with Gemini

After fetching content, you can ask Gemini (in the sidebar) questions about it!

**Tips:**
- Fetch a section, then ask Gemini to explain a concept
- Ask Gemini to create practice problems based on the content
- Use the test cases from LAB activities to check your solutions
- Ask Gemini to walk through participation activities step by step

In [ ]:
# Load a section and save it so Gemini can read it
md = fetch_section(7, 1, display_markdown=False)
with open("current_section.md", "w") as f:
    f.write(md)
print("Section saved to current_section.md - ask Gemini about it!")
display(Markdown(md))